In [ ]:
import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))

In [ ]:
import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file, path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)

In [ ]:
from glayout import MappedPDK,gf180
#from gdsfactory.cell import cell
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

In [ ]:
from glayout import pmos, nmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [ ]:
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route

In [ ]:
#########################################
# 6.     Instanciación de PDKs
#########################################
pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": False
}


In [ ]:
##########################################
# 7.    Instanciación de top_module
##########################################
current_limit_config = {
    "pdk": gf180,
    "layout_rules": {
        "spacing": gf180.util_max_metal_seperation() + 1,
        "routing_metal": "met2",
        "dummy_devices": False,
        "tie_layers": ("met2", "met1"),
        "sd_rmult": 1,
    },
}

pdk = gf180
top_comp = Component(name="Current_limit")
top_comp.name = "Current_limit"

rules = current_limit_config["layout_rules"]
spacing = rules["spacing"]
tie_layers = rules["tie_layers"]
sd_rmult = rules["sd_rmult"]

top_level = Component(name="current_limit")


In [ ]:
# =========================================================================
# 8. INSTANCIACIÓN DE DISPOSITIVOS
# =========================================================================
M0 = pmos(
    pdk,
    width=0.5,
    length=5.0,
    with_dummy=(False, False),
    with_substrate_tap=False,
    tie_layers=tie_layers,
    sd_rmult=sd_rmult,
    **pmos_kwargs
)
M1 = pmos(
    pdk,
    width=0.5,
    length=5.0,
    with_dummy=(False, False),
    with_substrate_tap=False,
    tie_layers=tie_layers,
    sd_rmult=sd_rmult,
    **pmos_kwargs
)

M2 = nmos(
    pdk,
    width=0.5,
    length=5.0,
    with_dummy=(False, False),
    with_substrate_tap=False,
    tie_layers=tie_layers,
    sd_rmult=sd_rmult,
    **nmos_kwargs
)

M0.name = "M0"
M1.name = "M1"
M2.name = "M1"


In [ ]:
# =========================================================================
# 9. COLOCACIÓN Y FLOORPLANNING 
# =========================================================================
from glayout.util.comp_utils import (
    align_comp_to_port,
    evaluate_bbox,
    prec_center,
    prec_ref_center,
)
M0_ref = top_comp << M0
M1_ref = top_comp << M1
M2_ref = top_comp << M2

M0_bbox = evaluate_bbox(M0)
M1_bbox = evaluate_bbox(M1)
M2_bbox = evaluate_bbox(M2)

M0_ref.move((0, 0))
M1_ref.move((0, -(M0_bbox[1] / 2 + M1_bbox[1] / 2 + spacing)))

M2_ref.rotate(90)
M2_ref.move((M1_ref.xmax + M2_bbox[0]/2 +  1.5*spacing, -(M0_bbox[1] / 2 + M1_bbox[1] / 2 + 0.25*spacing)))

display_component(top_comp, "./", scale=3)

In [ ]:
###################################################
# 10. Create a via stack from met2 to met3
###################################################
viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.

i_SUM_via = top_comp << viam2m3
ifwd_via = top_comp << viam2m3


In [ ]:
###################################################
# 11. Moving via stack from met2 to met3
###################################################

i_SUM_via.move(M0_ref.ports["multiplier_0_source_E"].center).movex(M1_bbox[1] / 3 + spacing)
ifwd_via.move(M1_ref.ports["multiplier_0_drain_E"].center).movex(M1_bbox[1] / 3 + spacing)

display_component(top_comp, "./", scale=3)

In [ ]:
import gdsfactory as gf
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle


In [ ]:

###################################################
# 12.b Internal Conections
###################################################
top_comp << straight_route( pdk, M0_ref.ports["tie_W_top_met_S"], M1_ref.ports["tie_W_top_met_N"])
top_comp << c_route(pdk, M1_ref.ports["source_W"], M0_ref.ports["gate_W"], extension=1.8)
top_comp << c_route(pdk, M0_ref.ports["gate_E"], M0_ref.ports["drain_E"], extension=1.5)
top_comp << L_route(pdk, M1_ref.ports["gate_E"], ifwd_via.ports["bottom_met_S"])
top_comp << straight_route(pdk, M2_ref.ports["source_S"], M2_ref.ports["gate_N"])
top_comp << straight_route(pdk, M2_ref.ports["gate_S"], M2_ref.ports["drain_N"])
top_comp << straight_route(pdk, M2_ref.ports["drain_N"], M2_ref.ports["tie_W_top_met_N"])

display_component(top_comp, "./", scale=3)

In [ ]:
# top_comp.add_ports(M0_ref.get_ports_list(), prefix="M0_")
# top_comp.add_ports(M1_ref.get_ports_list(), prefix="M1_")

# top_comp.add_ports(i_SUM_via.get_ports_list(), prefix="i_SUM_")
# top_comp.add_ports(ifwd_via.get_ports_list(), prefix="ifwd_")
# top_comp.add_ports(M2_ref.get_ports_list(), prefix="M2_")
display_component(top_comp, "./", scale=3)


In [ ]:
# =========================================================================
# 4. ADD AVDD POWER RAIL AND CONNECT SUPPLY NODES
# =========================================================================

# Calcular extensión horizontal para trazar el riel de alimentación
bbox = evaluate_bbox(top_comp)
top_y = top_comp.ymax + 3.0

# Trazar riel horizontal de metal2 para AVDD
avdd_rail = top_comp << gf.components.rectangle(
    size=(bbox[0] + 1, 1.0), 
    layer=pdk.get_layer("metal2")
)
avdd_rail.move(((M1_bbox[1] / 2) - 9, top_y))

via_avdd = top_comp << viam2m3
via_avdd.move((bbox[0] / 2 - 4.5, top_y + 0.5))

## METAL5 VSS TOP
bbox_vdd = evaluate_bbox(avdd_rail)

avdd_M5 = top_comp << gf.components.rectangle(
    size=(bbox_vdd[0]+18.14, 6.13), 
    layer=pdk.get_layer("metal5")
)
avdd_M5.move((-15, 6.8 + (6.13/2)))

#m5_label
avdd_label = top_comp << gf.components.rectangle(
    size=(bbox_vdd[0]+18.14, 6.13), 
    layer=pdk.get_layer("metal5_label")
)
avdd_label.move((-15, 6.8 + (6.13/2)))

avdd_M5_bot = top_comp << gf.components.rectangle(
    size=(bbox_vdd[0]+18.14, 6.13), 
    layer=pdk.get_layer("metal5")
)
avdd_M5_bot.move((-15, -14.8 - (6.13/2)))

#m5_label
avdd_label_bot = top_comp << gf.components.rectangle(
    size=(bbox_vdd[0]+18.14, 6.13), 
    layer=pdk.get_layer("metal5_label")
)
avdd_label_bot.move((-15, -14.8 - (6.13/2)))

display_component(top_comp, "./", scale=3)

In [ ]:
#met4_label VDD left
avdd1_label_m4 = top_comp << gf.components.rectangle(
    size=(5,27.5), #56.81
    layer=pdk.get_layer("metal4_label")
)
avdd1_label_m4.move((-15,-17.86+3.605)) #38

#met4_label VDD right
avdd2_label_m4 = top_comp << gf.components.rectangle(
    size=(5,27.5), #56.81
    layer=pdk.get_layer("metal4_label")
)
avdd2_label_m4.move((17.43 + 0.65,-17.86+3.605)) #38

display_component(top_comp, "./", scale=3)

In [ ]:
viam4m5 = via_stack(pdk, "met4", "met5", centered=True)

via_avdd_M5 = top_comp << viam4m5
via_avdd_M5.move((-12.5,16-2.5))

via_avdd_M5B = top_comp << viam4m5
via_avdd_M5B.move((-12.5,-16+1.5))

via_avdd_M5C = top_comp << viam4m5
via_avdd_M5C.move((18.29+2.29,16-2.5))

via_avdd_M5D = top_comp << viam4m5
via_avdd_M5D.move((18.29+2.29,-16+1.5))

top_comp  << straight_route(pdk, via_avdd_M5.ports["bottom_met_S"], via_avdd_M5B.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")
top_comp  << straight_route(pdk, via_avdd_M5.ports["bottom_met_S"], via_avdd_M5B.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")
top_comp  << straight_route(pdk, via_avdd_M5C.ports["bottom_met_S"], via_avdd_M5D.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")
top_comp  << straight_route(pdk, via_avdd_M5C.ports["bottom_met_S"], via_avdd_M5D.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")

viam2m5 = via_stack(pdk, "met2", "met5", centered=True)
avdd_M5_PIN = top_comp << viam2m5
avdd_M5_PIN.move((0,16-(6.13/2)))

top_comp << L_route(pdk,avdd_M5_PIN.ports["bottom_met_S"], via_avdd.ports["bottom_met_W"])
top_comp << L_route(pdk, M0_ref.ports["tie_W_top_met_N"], via_avdd.ports["bottom_met_W"])


display_component(top_comp, "./", scale=2)


In [ ]:
bbox1 = evaluate_bbox(top_comp)


avss_M5 = top_comp << gf.components.rectangle(
    size=(bbox1[0]+15, 5), 
    layer=pdk.get_layer("metal5")
)
avss_M5.move((-(bbox1[0]/2)-3.5, bbox1[1]/2 + spacing))

avss_M5B = top_comp << gf.components.rectangle(
    size=(bbox1[0]+15, 5), 
    layer=pdk.get_layer("metal5")
)
avss_M5B.move((-(bbox1[0]/2)-3.5, -bbox1[1]/2 - spacing - 6.87))



avss_M5L = top_comp << gf.components.rectangle(
    size=(bbox1[0]+15, 5), 
    layer=pdk.get_layer("metal5_label")
)
avss_M5L.move((-(bbox1[0]/2)-3.5, bbox1[1]/2 + spacing))

avss_M5BL = top_comp << gf.components.rectangle(
    size=(bbox1[0]+15, 5), 
    layer=pdk.get_layer("metal5_label")
)
avss_M5BL.move((-(bbox1[0]/2)-3.5, -bbox1[1]/2 - spacing - 6.87))

bbox_d = evaluate_bbox(avss_M5BL)

via_avss_M5 = top_comp << viam4m5
via_avss_M5.move((-20.04,bbox1[1]/2 + spacing + 2.5))

via_avss_M5B = top_comp << viam4m5
via_avss_M5B.move((28.04,bbox1[1]/2 + spacing + 2.5))

via_avss_M5C = top_comp << viam4m5
via_avss_M5C.move((-20.04,-bbox1[1]/2 - spacing - 4.37))

via_avss_M5D = top_comp << viam4m5
via_avss_M5D.move((28.04,-bbox1[1]/2 - spacing - 4.37))

top_comp  << straight_route(pdk, via_avss_M5.ports["bottom_met_S"], via_avss_M5C.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")
top_comp  << straight_route(pdk, via_avss_M5B.ports["bottom_met_S"], via_avss_M5D.ports["bottom_met_N"],glayer1="met4",glayer2="met4",width="5")

avss_M4L = top_comp << gf.components.rectangle(
    size=(5, 42.83), 
    layer=pdk.get_layer("metal4_label")
)
avss_M4L.move((25.54, -22.35))

avss_M4BL = top_comp << gf.components.rectangle(
    size=(5, 42.83), 
    layer=pdk.get_layer("metal4_label")
)
avss_M4BL.move((-22.54, -22.35))


avss_M5_PIN = top_comp << viam2m5
avss_M5_PIN.move((4,-22.6))

top_comp << L_route(pdk,avss_M5_PIN.ports["bottom_met_W"], M2_ref.ports["drain_E"])


display_component(top_comp, "./", scale=1)


In [ ]:
#----------- Adding snippet code to move all the placed components to the proper position, 
#----------- in order to set origin into 0,0
# Place this snippet after all your material polygones are placed, and before your ports are placed
bbox_all = top_comp.bbox[0]
for ref in top_comp.references:
    ref.movex(-1*bbox_all[0]).movey(-1*bbox_all[1])
display_component(top_comp, "./", scale=1)

In [ ]:

# =========================================================================
# 5. ADD PORTS AND PHYSICAL PIN LABELS (done BEFORE routing)
# =========================================================================

psize = (0.5, 0.5)

# i_SUM
box_isum = evaluate_bbox(i_SUM_via)
center_S_i_SUM = i_SUM_via.ports["bottom_met_S"].center

i_SUM_port = top_comp << gf.components.rectangle(size=psize, layer=pdk.get_layer("metal2"))
i_SUM_port.move(i_SUM_via.ports["bottom_met_S"].center)
top_comp.add_port(
    "i_SUM",
    center= (center_S_i_SUM[0], center_S_i_SUM[1] + box_isum[1]/2), # Move the port up by half the height of the bounding box plus an offset
    width=1.0,
    orientation=180,
    layer=pdk.get_layer("metal2"),
    port_type="electrical"
)
top_comp.add_label(
    text="i_SUM",
    position=(center_S_i_SUM[0], center_S_i_SUM[1] + box_isum[1]/2),
    layer=pdk.get_glayer("met2_label"),
    magnification=1
)

# ifwd
box_ifwd = evaluate_bbox(ifwd_via)
center_S_ifwd = ifwd_via.ports["bottom_met_S"].center

ifwd_port = top_comp << gf.components.rectangle(size=psize, layer=pdk.get_layer("metal2"))
ifwd_port.move(ifwd_via.ports["bottom_met_S"].center)

top_comp.add_port(
    "ifwd",
    center=(center_S_ifwd[0], center_S_ifwd[1] + box_ifwd[1]/2),
    width=1.0,
    orientation=180,
    layer=pdk.get_layer("metal2"),
    port_type="electrical"
)
top_comp.add_label(
    text="ifwd",
    position=(center_S_ifwd[0], center_S_ifwd[1] + box_ifwd[1]/2),
    layer=pdk.get_glayer("met2_label"),
    magnification=1
)

###################################################
# 12.a Pin Conections
###################################################
top_comp << straight_route(pdk, M0_ref.ports["multiplier_0_source_E"], top_comp.ports["i_SUM"])
top_comp << straight_route(pdk, M1_ref.ports["multiplier_0_drain_E"], top_comp.ports["ifwd"])


# avdd (bulk tie, via avdd1)

avdd_port = top_comp << gf.components.rectangle(size=psize, layer=pdk.get_layer("metal5"))
avdd_port.move(avdd_M5_PIN.ports["top_met_S"].center)



top_comp.add_port(
    "avdd",
    center=avdd_M5_PIN.ports["top_met_S"].center,
    width=1.0,
    orientation=180,
    layer=pdk.get_layer("metal5"),
    port_type="electrical"
)
top_comp.add_label(
    text="avdd",
    position=avdd_M5_PIN.ports["top_met_S"].center,
    layer=pdk.get_glayer("met5_label"),
    magnification=1
)
################################################################################################

avss_port = top_comp << gf.components.rectangle(size=psize, layer=pdk.get_layer("metal5"))
avss_port.move(avss_M5_PIN.ports["top_met_S"].center)

top_comp.add_port(
    "avss",
    center=avss_M5_PIN.ports["top_met_S"].center,
    width=1.0,
    orientation=180,
    layer=pdk.get_layer("metal5"),
    port_type="electrical"
)
top_comp.add_label(
    text="avss",
    position=avss_M5_PIN.ports["top_met_S"].center,
    layer=pdk.get_glayer("met5_label"),
    magnification=1
)

display_component(top_comp, "./", scale=3)


In [ ]:
bbox1 = evaluate_bbox(top_comp)
boundary = top_comp << gf.components.rectangle(
    size=(bbox1[0], bbox1[1]), 
    layer=(0,0)
)
boundary.move((top_comp.xmin, top_comp.ymin))
display_component(top_comp, "./", scale=3)

In [ ]:
evaluate_bbox(top_comp)

In [ ]:
evaluate_bbox(top_comp)[0]*evaluate_bbox(top_comp)[1]


In [ ]:
top_comp.name="Current_limit"
drc_result = gf180.drc_magic(top_comp, top_comp.name)
print("Making sure your environment varriables are still correctly set")
print(" I am using PDK in: ",os.environ['PDK_ROOT'],"\n PDK name: ",os.environ['PDK'],"\n The PDK files are at: ",os.environ['PDKPATH'])

In [ ]:
top_comp.unlock()

In [ ]:
import os
from pathlib import Path
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
design_name = top_comp.name
path_to_dir = "/foss/designs/libs/snn_analog/Current_limit/"

pex_path = path_to_dir + f"{design_name}.spice"
gds_path = path_to_dir + f"{design_name}.gds"

top_comp.write_gds(str(gds_path))
    
magic_script_content = f"""
drc off            
gds flatglob *\\$\\$*
gds read {gds_path}

flatten {design_name}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {str(pex_path)}
exit
"""



with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
    magic_script_file.write(magic_script_content)
    magic_script_path = magic_script_file.name
    
magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
magic_subproc = subprocess.run(
    magic_cmd, 
    shell=True,
    check=True,
    capture_output=True
)

magic_subproc_code = magic_subproc.returncode
magic_subproc_out = magic_subproc.stdout.decode('utf-8')
print(magic_subproc_out)

import glob
extensions = [
            "els"
            "*.gds",
            "*.ext",
            "*.res.ext",
            "*.lvs.rpt",
            "*_lvs.rpt",
            "*.nodes",
            "*.sim",
            "*.pex.spice",
            "*_pex.spice"
            ]
files_to_delete = []
for ext in extensions:
    files_to_delete.extend(glob.glob(ext))
    
# Delete the files
for file_path in files_to_delete:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

In [ ]:
gf180.lvs_netgen(
    layout=top_comp,
    design_name = top_comp.name,
    pdk_root = Path(os.environ['PDKPATH']),
    lvs_setup_tcl_file = Path(os.environ['PDKPATH']) / "libs.tech" / "netgen" / f"{os.environ['PDK']}_setup.tcl",
    lvs_schematic_ref_file = Path(str(path_to_dir + "Current_limit.spice")),
    netlist = Path(str(path_to_dir + "Current_limitSCH.spice")),
    output_file_path =  Path(str(path_to_dir))
)



In [ ]:
from pathlib import Path
import os
import subprocess

pdk_name = os.environ.get("PDK", "gf180mcuD")
pdk_root = Path(os.environ.get("PDK_ROOT", ""))
pdk_path_env = Path(os.environ.get("PDKPATH", ""))

# Resolve the correct magicrc path even if PDKPATH is stale or duplicated.
candidate_pdkpaths = [
    pdk_path_env,
    pdk_root / pdk_name,
    pdk_root,
]
magicrc_file = None
for candidate in candidate_pdkpaths:
    if not str(candidate):
        continue
    rc = candidate / "libs.tech" / "magic" / f"{pdk_name}.magicrc"
    if rc.exists():
        magicrc_file = rc
        break

if magicrc_file is None:
    raise FileNotFoundError(
        "No se encontro el magicrc. Revisar PDK_ROOT/PDK/PDKPATH. "
        f"Busque en: {[str(c) for c in candidate_pdkpaths if str(c)]}"
    )

# --- Cambio clave: usar path_to_dir en vez de Path.cwd()/"exports" ---
path_to_dir = "/foss/designs/CapiMagics/designs/libs/snn_analog/Current_limit/"
output_dir = Path(path_to_dir)
output_dir.mkdir(parents=True, exist_ok=True)

design_name = top_comp.name
gds_path = output_dir / f"{design_name}.gds"
lef_path = output_dir / f"{design_name}.lef"

top_comp.write_gds(str(gds_path))

magic_script = f"""
drc off
gds flatglob *\\$\\$*
gds read {gds_path}
load {design_name}
lef write {lef_path}
quit -noprompt
"""

result = subprocess.run(
    ["magic", "-rcfile", str(magicrc_file), "-noconsole", "-dnull"],
    input=magic_script,
    text=True,
    capture_output=True,
    check=True,
)

print(f"CWD kernel: {Path.cwd()}")
print(f"LEF generado: {lef_path}")
print(f"LEF existe: {lef_path.exists()}")
if lef_path.exists():
    print("\nCabecera LEF:")
    with open(lef_path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(15):
            line = f.readline()
            if not line:
                break
            print(line.rstrip())
else:
    print("\n--- STDOUT COMPLETO DE MAGIC (para depurar) ---")
    print(result.stdout)
    print("\n--- STDERR COMPLETO DE MAGIC ---")
    print(result.stderr)